In [1]:
import numpy as np
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

from sklearn.compose import ColumnTransformer
from imblearn.pipeline import Pipeline as ImbPipeline

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_auc_score
from sklearn import set_config

# For experiment analysis
import itertools
import statistics
import matplotlib.pyplot as plt

import exp as exptbl

In [2]:
categorical_features = ["workclass", "marital-status", "occupation", "relationship", "race", "native-country"]
ordinal_features = ['age', 'hours-per-week', "education-num"]
binary_features = ['sex']

In [3]:
# Load and begin prepping strip search data
df = pd.read_csv('./datasets/torontostripsearch.csv', delimiter=',')

df['IsYouth'] = np.where(df['Youth_at_arrest__under_18_years'] == "Not a youth", 0, 1)
df = df.rename(columns={"Arrest_Month" : "Arrest_Quarter"})
df.drop(columns=df.columns.intersection(['SearchReason_CauseInjury', 'SearchReason_AssistEscape', 'SearchReason_PossessWeapons', 'Arrest_Year',
                                         'SearchReason_PossessEvidence', 'Youth_at_arrest__under_18_years', "_defensive_or_escape_risk",
                                         'ObjectId', 'EventID', 'ArrestID', 'PersonID', 'Booked', 'ItemsFound']), axis=1, inplace=True)

df.replace({'Perceived_Race': {np.nan: 'Unknown or Legacy'}}, inplace=True)

df = df.drop(df[df['Sex'] == 'U'].index)

df.dropna(how="any", inplace=True)
categorical_features = ['Perceived_Race', 'Sex', 'Occurrence_Category', 'ArrestLocDiv']
ordinal_features = ['Age_group__at_arrest_', 'Arrest_Quarter']
binary_features = ['IsYouth','Actions_at_arrest___Concealed_i', 'Actions_at_arrest___Combative__',
                  'Actions_at_arrest___Resisted__d', 'Actions_at_arrest___Mental_inst',
                  'Actions_at_arrest___Assaulted_o', 'Actions_at_arrest___Cooperative']


X = df.drop('StripSearch', axis=1)
X = X.drop('IsYouth', axis=1)
y = df['StripSearch']

# Map the age groups to the specified values
custom_mapping = {
    'Aged 17 years and under': 0,
    'Aged 17 years and younger': 0,
    'Aged 18 to 24 years': 1,
    'Aged 25 to 34 years': 2,
    'Aged 35 to 44 years': 3,
    'Aged 45 to 54 years': 4,
    'Aged 55 to 64 years': 5,
    'Aged 65 and older': 6,
    'Aged 65 years and older': 6
}

quarter_mapping = {
"Jan-Mar" : 0,
"Apr-June" : 1,
"July-Sept" : 3,
"Oct-Dec" : 4
}

X['Age_group__at_arrest_'] = X['Age_group__at_arrest_'].map(custom_mapping)
X['Arrest_Quarter'] = X['Arrest_Quarter'].map(quarter_mapping)


rev_custom_mapping = {
    0 : 'Aged 17 years and younger',
    1 : 'Aged 18 to 24 years'      ,
    2 : 'Aged 25 to 34 years'      ,
    3 : 'Aged 35 to 44 years'      ,
    4 : 'Aged 45 to 54 years'      ,
    5 : 'Aged 55 to 64 years'      ,
    6 : 'Aged 65 and older'        
}

rev_quarter_mapping = {
    0 : "Jan-Mar"   ,
    1 : "Apr-June"  ,
    3 : "July-Sept" ,
    4 : "Oct-Dec"   
}

custom_encoders = {
    'Age_group__at_arrest_': rev_custom_mapping,
    'Arrest_Quarter': rev_quarter_mapping
}



In [4]:
# Create train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


# Apply label encoding for categorical features
encoders = {}
for feature in categorical_features + ordinal_features:
    le = LabelEncoder()
    X_train[feature] = le.fit_transform(X_train[feature])
    X_test[feature] = le.transform(X_test[feature])
    print(feature)
    for index, label in enumerate(le.classes_):
      print(f"Label {index}: {label}")
    encoders[feature] = le

      

Perceived_Race
Label 0: Black
Label 1: East/Southeast Asian
Label 2: Indigenous
Label 3: Latino
Label 4: Middle-Eastern
Label 5: South Asian
Label 6: Unknown or Legacy
Label 7: White
Sex
Label 0: F
Label 1: M
Occurrence_Category
Label 0: Assault
Label 1: Assault & Other crimes against persons
Label 2: Break & Enter
Label 3: Break and Enter
Label 4: Crimes against Children
Label 5: Drug Related
Label 6: FTA/FTC, Compliance Check & Parollee
Label 7: FTA/FTC/Compliance Check/Parollee
Label 8: Fraud
Label 9: Harassment & Threatening
Label 10: Harassment/Threatening
Label 11: Homicide
Label 12: Impaired
Label 13: LLA
Label 14: Mental Health
Label 15: Mischief
Label 16: Mischief & Fraud
Label 17: Other Offence
Label 18: Other Statute
Label 19: Other Statute & Other Incident Type
Label 20: Police Category - Administrative
Label 21: Police Category - Incident
Label 22: Robbery & Theft
Label 23: Robbery/Theft
Label 24: Sexual Related Crime
Label 25: Sexual Related Crimes & Crimes Against Childr

In [5]:
# Train xgboost classifier from sklearn on it
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ordinal_features)], remainder='passthrough')

xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.2, max_depth=5, gamma = 0, min_child_weight=5, subsample=1,
                          eval_metric = "logloss", random_state=42)

xgb_model = ImbPipeline([
    ('preprocessor', preprocessor),
    ('classifier', xgb_model)])

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

print(f'XGBoost Test Accuracy: {accuracy_score(y_test, y_pred_xgb):.3f}')
print(f'XGBoost Test F1: {f1_score(y_test, y_pred_xgb):.3f}')


XGBoost Test Accuracy: 0.900
XGBoost Test F1: 0.474


In [6]:
y_test.shape

(13016,)

In [7]:
y_pred_xgb.shape

(13016,)

In [8]:
def label_false_positives(X_test, y_test, y_pred):
    fps = np.zeros(y_pred.shape[0])
    y_truth = y_test.copy().reset_index(drop=True)
    test_copy = X_test.copy() 
    for i in test_copy.reset_index(drop=True).itertuples():
        if y_pred[i[0]] == 1 and y_truth[i[0]] == 0:
            fps[i[0]] = 1
    test_copy['targetcol'] = np.round(fps, 2)
    return test_copy
    

In [9]:
def label_false_negatives(X_test, y_test, y_pred):
    fps = np.zeros(y_pred.shape[0])
    y_truth = y_test.copy().reset_index(drop=True)
    test_copy = X_test.copy() 
    for i in test_copy.reset_index(drop=True).itertuples():
        if y_pred[i[0]] == 0 and y_truth[i[0]] == 1:
            fps[i[0]] = 1
    test_copy['targetcol'] = np.round(fps, 2)
    return test_copy
    

In [10]:
x_res = label_false_positives(X_test, y_test, y_pred_xgb)
x_res['targetcol'].sum()

326.0

In [11]:
min_sup = 0.1
exp_table_cols = ['Perceived_Race', 'Sex','Age_group__at_arrest_', 'targetcol']
x_res[exp_table_cols].to_csv("temp.csv", index=False)
pd.DataFrame().to_csv("target.csv", index=False)
res = exptbl.calculate_table(f"target.csv", "temp.csv", f"target.csv", min_support_param=min_sup)


Compiling with commands:  ['g++', 'c:\\Users\\Andrew\\Documents\\School\\6320 AI Fairness\\Homework 1\\exp\\Explanations.cpp', 'c:\\Users\\Andrew\\Documents\\School\\6320 AI Fairness\\Homework 1\\exp\\Lighthouse.cpp', '-o', 'program']
Arguments to the program: target.csv temp.csv 3 15 0 target.csv 0.1
Time: 0:00:09.633511


In [14]:
# read in the new explanation table
exp_table = pd.read_csv(res, sep=";")
exp_table

,Perceived_Race,Sex,Age_group__at_arrest_,targetcol,support
0,*,*,*,0.025,13016
1,0,1,*,0.034,2864
2,7,1,*,0.025,4273
3,7,*,2,0.038,1687
4,*,*,3,0.029,3210
5,7,*,3,0.032,1511
6,*,1,1,0.028,1598
7,*,*,2,0.030,4237
8,*,1,4,0.018,1479
9,7,*,*,0.027,5501
